<a href="https://colab.research.google.com/github/mr-osiris/pothole-detect/blob/main/CV_Mini_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import shutil

dataset_path = '/content/drive/MyDrive/PotholeDataset'

train_images_path = os.path.join(dataset_path, 'train/images')
train_labels_path = os.path.join(dataset_path, 'train/labels')
val_images_path = os.path.join(dataset_path, 'valid/images')
val_labels_path = os.path.join(dataset_path, 'valid/labels')

for split in ['train', 'val']:
    os.makedirs(f'data/{split}/images', exist_ok=True)
    os.makedirs(f'data/{split}/labels', exist_ok=True)

for f in os.listdir(train_images_path):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        shutil.copy(os.path.join(train_images_path, f), f'data/train/images/')
for f in os.listdir(train_labels_path):
    if f.lower().endswith('.txt'):
        shutil.copy(os.path.join(train_labels_path, f), f'data/train/labels/')

for f in os.listdir(val_images_path):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        shutil.copy(os.path.join(val_images_path, f), f'data/val/images/')
for f in os.listdir(val_labels_path):
    if f.lower().endswith('.txt'):
        shutil.copy(os.path.join(val_labels_path, f), f'data/val/labels/')

print("Dataset copied from Drive to local Colab environment.")


Dataset copied from Drive to local Colab environment.


In [ ]:
import yaml

data_yaml = {
    'train': os.path.abspath('data/train/images'),
    'val': os.path.abspath('data/val/images'),
    'nc': 1,
    'names': ['pothole']
}

os.makedirs('data', exist_ok=True)

with open('data/pothole.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print("YOLOv8 dataset YAML config created with absolute paths:")
print(data_yaml)


YOLOv8 dataset YAML config created with absolute paths:
{'train': '/content/data/train/images', 'val': '/content/data/val/images', 'nc': 1, 'names': ['pothole']}


In [ ]:
!pip install ultralytics opencv-python matplotlib scikit-learn requests PyYAML --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 50.2 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

print("Starting training...")
model.train(data='data/pothole.yaml', epochs=10, imgsz=320, batch=16, save=True, name='yolov8-pothole')
print("Training completed.")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting training...
Ultralytics 8.3.227 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/pothole.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=

In [ ]:
import matplotlib.pyplot as plt
import time
from IPython.display import display, clear_output

def simulate_suspension_response():
    print("[ALERT] Pothole detected! Suspension adjusted.")
    fig, ax = plt.subplots(figsize=(4,2))
    plt.ion()

    line, = ax.plot([0, 1], [0, 0], lw=8, color='blue')
    ax.set_ylim(-1, 1)
    ax.set_title("Active Suspension Simulation")
    display(fig)

    for i in range(10):
        line.set_ydata([0, 0.1 * i])
        fig.canvas.draw()
        fig.canvas.flush_events()
        time.sleep(0.05)
        clear_output(wait=True)
        display(fig)

    for i in range(10, -1, -1):
        line.set_ydata([0, 0.1 * i])
        fig.canvas.draw()
        fig.canvas.flush_events()
        time.sleep(0.05)
        clear_output(wait=True)
        display(fig)

    plt.close(fig)
    clear_output(wait=True)


In [ ]:
import os
from ultralytics import YOLO

def find_weights():
    for folder in ['runs/train', 'runs/detect']:
        for root, dirs, files in os.walk(folder):
            if 'best.pt' in files:
                return os.path.join(root, 'best.pt')
            if 'last.pt' in files:
                last_path = os.path.join(root, 'last.pt')
    return last_path if 'last_path' in locals() else None

weight_path = find_weights()
assert weight_path is not None, "No trained YOLOv8 weights found. Complete training first!"

print("Found trained model weights:", weight_path)
model = YOLO(weight_path)


Found trained model weights: runs/detect/yolov8-pothole/weights/best.pt


In [ ]:
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, HTML
from google.colab import files
import os

def process_video_with_output(model):
    """
    Process user-uploaded video, detect potholes, and save output with bounding boxes
    """
    print("=" * 60)
    print("VIDEO UPLOAD FOR POTHOLE DETECTION")
    print("=" * 60)
    print("\nPlease upload a video file for pothole detection...")

    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Exiting.")
        return

    input_video_path = list(uploaded.keys())[0]
    print(f"\n✓ Video uploaded: {input_video_path}")

    cap = cv2.VideoCapture(input_video_path)

    if not cap.isOpened():
        print("Error: Could not open video file.")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"\nVideo Info:")
    print(f"  Resolution: {width}x{height}")
    print(f"  FPS: {fps}")
    print(f"  Total Frames: {total_frames}")

    output_video_path = 'pothole_detected_output.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    print(f"\n{'='*60}")
    print("PROCESSING VIDEO - DETECTING POTHOLES...")
    print(f"{'='*60}\n")

    frame_count = 0
    pothole_detections = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        results = model(frame, verbose=False)

        frame_with_detections = results[0].plot()

        num_detections = len(results[0].boxes)
        if num_detections > 0:
            pothole_detections += 1

        out.write(frame_with_detections)

        if frame_count % 30 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"Progress: {progress:.1f}% | Frame {frame_count}/{total_frames} | Potholes detected in {pothole_detections} frames")

        if frame_count % 10 == 0:
            frame_rgb = cv2.cvtColor(frame_with_detections, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10, 6))
            plt.imshow(frame_rgb)
            plt.axis('off')
            plt.title(f"Frame {frame_count}/{total_frames} - Detections: {num_detections}")
            display(plt.gcf())
            clear_output(wait=True)
            plt.close()

    cap.release()
    out.release()

    print(f"\n{'='*60}")
    print("PROCESSING COMPLETE!")
    print(f"{'='*60}")
    print(f"\nResults:")
    print(f"  Total frames processed: {frame_count}")
    print(f"  Frames with pothole detections: {pothole_detections}")
    print(f"  Detection rate: {(pothole_detections/frame_count)*100:.2f}%")
    print(f"\n✓ Output video saved as: {output_video_path}")

    print("\nDownloading output video...")
    files.download(output_video_path)
    print("✓ Download complete!")

    return output_video_path

print("\nStarting Enhanced Pothole Detection System...")
print("This will:")
print("  1. Ask you to upload a video")
print("  2. Process it with the trained YOLOv8 model")
print("  3. Draw bounding boxes around detected potholes")
print("  4. Save and download the output video\n")

output_path = process_video_with_output(model)


PROCESSING COMPLETE!

Results:
  Total frames processed: 692
  Frames with pothole detections: 582
  Detection rate: 84.10%

✓ Output video saved as: pothole_detected_output.mp4



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download complete!
